In [1]:
import requests
from urllib.parse import urlencode

import pandas as pd
import json

BASE_API_URL = 'https://api.gotriple.eu/api/'

In [ ]:
language = 'en'
provider = 'base'

params = {
    'q': '',
    'fq': {
        'type': 'typ_article',
        'has_pdf': 'true',
        'in_language': language,
        # 'provider': provider,
        'topic': 'archeo',
    },
    'include_duplicates': 'false',
    'aggs': {},
    'page': 1,
    'size': 100, # max 100
}

In [18]:
def build_api_url(params):
    cleaned_params = {}
    for key, value in params.items():
        if value:
            if isinstance(value, dict):
                if key == 'fq':
                    cleaned_params[key] = urlencode(value).replace('&', ';')
                elif key == 'aggs':
                    cleaned_params[key] = ';'.join([k + ','+ v for k,v in value.items()])
            else:
               cleaned_params[key] = value
    return urlencode(cleaned_params)

In [ ]:
def get_response(query, params):
    params['q'] = query
    endpoint = 'documents'
    objects_list = []
    url = f'{BASE_API_URL}{endpoint}?{build_api_url(params)}'
    response = requests.get(url)
    print('Response length: ', response.json().get('meta').get('total'))
    while True:
        response = requests.get(url)
        if not response.ok:
            print('Response error:', response.status_code)
            print(response.text)
            break
        if not response.json()['data']:
            print('Empty "data" field. End of process.')
            break
        else:
            objects_list.extend(response.json()['data'])
            print(url)
            params['page'] += 1
            if params['page'] == 100:
                print('100 pages reached. End of process.')
                break
            url = f'{BASE_API_URL}{endpoint}?{build_api_url(params)}'

    print(len(objects_list))
    return objects_list

In [ ]:
objects_list = get_response(''. params)
with open(f'gotriple_response_{language}.json', 'w', encoding='utf-8') as jfile:
    json.dump(objects_list, jfile, indent=4, ensure_ascii=False)

In [ ]:
to_df = []
for obj in objects_list:
    object_id = obj.get('id')

    # in_language - gotriple field
    in_language = ' | '.join(obj.get('in_language'))

    # original_languages
    original_languages = ' | '.join(obj.get('original_languages'))

    # date_published
    date_published = obj.get('date_published')

    # year
    year = date_published[:4] if date_published else ''

    # abstract_present
    abstract_present = True if obj.get('abstract') else False

    # abstract_len
    abstract_len = len(obj.get('abstract')[0].get('text')) if abstract_present else 0

    # keywords_present
    keywords_present = True if obj.get('keywords') else False

    # keyword_count
    keywords_count = len(obj.get('keywords')) if keywords_present else 0

    # source
    source = ' | '.join(obj.get('provider'))

    to_df.append((object_id, in_language, original_languages, date_published, year, abstract_present, abstract_len, keywords_present, keywords_count, source))

df = pd.DataFrame(to_df, columns=['object_id', 'in_language', 'original_languages', 'date_published', 'year', 'abstract_present', 'abstract_len', 'keywords_present', 'keywords_count', 'source'])

df.to_csv(f'gotriple_response_{language}.csv', index=False)